# Figure 4

In [1]:
import numpy as np
import pickle
from flax import nnx
from pathlib import Path
import string
import jax
import sys
from jax import numpy as jnp
from jax import random as jrand
from jax import lax, jit
from flax import nnx
from functools import partial
import gc
import time
from tqdm.auto import tqdm

# plot related
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.patches import Rectangle
from matplotlib.ticker import FormatStrFormatter

plt.rcParams["svg.fonttype"] = "none"

# model related
from nntp.utils.datatype import load_class
from nntp.utils.model import restore_model_from_checkpoint
from nntp.utils.plot import find_experiments, load_experiments_parallel
from nntp.datasets import DataSetManager

src_path = (Path.cwd().parent / "src").resolve()
sys.path.insert(0, str(src_path))
from CoSynRNNModel.CoSynRNNModel import CoSynRNN
from entrypoint import register

register()

# 注册完成后再导出运行时 Enum
from nntp.datasets import TaskRegistry
from nntp.models import ModelRegistry
from nntp.schema import create_root_config

TASKS = TaskRegistry.export_registry()
TASKS_KEY = TaskRegistry.export_registry_key()

MODELS = ModelRegistry.export_registry()
MODELS_KEY = ModelRegistry.export_registry_key()
MODELS_CONFIG = ModelRegistry.export_registry_config()

from common import (
    subtitle_fontsize,
    panel_indexing_fontsize,
    feature_size,
    output_size,
    CHANNEL,
    CHANNEL_NAME_MAPPING,
    COLORS,
)

/home/ivan.y.gao/miniconda3/envs/nntp-cuda12-seaborn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset_prefix = "runtime/data"
dataset_postfix = "-ry-seed42-train1024-validation1024-test1024.npz"
task_names = [
    "fdgo",
    "reactgo",
    "delaygo",
    "fdanti",
    "reactanti",
    "delayanti",
    "dm1",
    "dm2",
    "contextdm1",
    "contextdm2",
    "multidm",
    "delaydm1",
    "delaydm2",
    "contextdelaydm1",
    "contextdelaydm2",
    "multidelaydm",
]

root_path = Path("..").expanduser().resolve()
datasets = {}
for task_name in task_names:
    target_dataset_path = f"{task_name}{dataset_postfix}"
    dataset = DataSetManager.load(
        f"{task_name}-ry", 42, 1024, 1024, 1024, root_path / dataset_prefix
    )
    datasets[dataset.key] = dataset
    print(dataset.key)

tasks = {}
for dataset in datasets.values():
    task = TASKS[TASKS_KEY[dataset.key].value].value
    if isinstance(task.decoder, str):
        task.decoder = load_class(task.decoder)
    if isinstance(task.evaluator, str):
        task.evaluator = load_class(task.evaluator)
    tasks[dataset.key] = task

fdgo-ry
reactgo-ry
delaygo-ry
fdanti-ry
reactanti-ry
delayanti-ry
dm1-ry
dm2-ry
contextdm1-ry
contextdm2-ry
multidm-ry
delaydm1-ry
delaydm2-ry
contextdelaydm1-ry
contextdelaydm2-ry
multidelaydm-ry


In [3]:
def parse_similarity_name(name):
    prefixes = ("similarity_", "baseline_")
    prefix = next(
        (prefix for prefix in prefixes if name.startswith(prefix)),
        None,
    )

    if prefix is None:
        raise ValueError(f"Invalid experiment name: {name}")

    try:
        task1, task2 = name.removeprefix(prefix).split("_", maxsplit=1)
    except ValueError as error:
        raise ValueError(f"Invalid experiment name: {name}") from error

    return [task1, task2]


# create path
out_path = Path("./output")
out_path.mkdir(parents=True, exist_ok=True)
root_path = Path("..").expanduser().resolve() / "figures/similarity"
experiments = find_experiments(root_path, parse_similarity_name)
for path in experiments:
    print(path)

# load data
summary_path = root_path / "summary.blob"
if summary_path.exists():
    with open(summary_path, "rb") as f:
        loaded_experiments = pickle.load(f)

    print(
        f"Loaded {len(loaded_experiments)} completed experiments "
        f"from {summary_path}"
    )
else:
    loaded_experiments = []
    print("No existing summary found; starting from scratch.")

    config = pickle.load(
        open(experiments[0]["paths"][0] / "metadata_config.blob", "rb")
    )
    config["angle_tolerance"] = 36.0
    config["angle_threshold"] = 36.0
    AbstractModel = CoSynRNN(nnx.Rngs(42), feature_size, output_size, config)
    for task in tasks.values():
        task.evaluator = task.evaluator(config)

    for experiment in tqdm(experiments, desc="Experiments"):
        Ts = experiment["tasks"]
        extra = []
        acc1 = []
        acc2 = []

        for path in tqdm(experiment["paths"], desc=f"Loading {Ts}", leave=False):
            is_baseline = Ts[0] == Ts[1]
            model = restore_model_from_checkpoint(
                path,
                AbstractModel,
                200 if not is_baseline else 100,
            )

            # ================= extra neurons =================
            W_rec_active_synapse = jnp.abs(model.W_rec) > model.threshold

            recurrent_free_neurons = (
                (W_rec_active_synapse.sum(axis=0) == 0)
                & (W_rec_active_synapse.sum(axis=1) == 0)
                & model.trainable_mask
            )

            cue = int(jax.device_get(model.cue.get_value()))

            readout_free_neurons = (
                jnp.sum(
                    jnp.abs(
                        model.W_out * model.modulation_activation(model.W_mask[cue])
                    )
                    > model.threshold,
                    axis=-1,
                )
                == 0
            )

            extra_neuron = (
                256
                - jnp.sum(recurrent_free_neurons & readout_free_neurons)
                - (jnp.sum(model.identity == 0) if not is_baseline else 0)
            )
            extra_neuron = int(jax.device_get(extra_neuron))
            extra.append(extra_neuron)

            # ================= task-1 accuracy =================
            task_name = Ts[0]
            task = tasks[task_name]
            X, Y, M = (
                datasets[task_name].test_x,
                datasets[task_name].test_y,
                datasets[task_name].test_mask,
            )
            y, _ = model.evaluation(X)
            metrics1 = task.evaluator(y, Y, M)
            metrics1 = jax.device_get(metrics1)
            acc1.append(metrics1["accuracy_angle"])

            # ================= task-2 accuracy =================
            task_name = Ts[1]
            task = tasks[task_name]
            X, Y, M = (
                datasets[task_name].test_x,
                datasets[task_name].test_y,
                datasets[task_name].test_mask,
            )
            y, _ = model.evaluation(X)
            metrics2 = task.evaluator(y, Y, M)
            metrics2 = jax.device_get(metrics2)

            # 结果已经拷回 CPU，现在可以删除 GPU 对象
            del y
            del model
            del W_rec_active_synapse
            del recurrent_free_neurons
            del readout_free_neurons

            gc.collect()

            acc2.append(metrics2["accuracy_angle"])

        loaded_experiments.append(
            {
                "tasks": Ts,
                "data": {
                    "extra_neuron": extra,
                    "accuracy1": acc1,
                    "accuracy2": acc2,
                },
            }
        )

    with open(summary_path, "wb") as f:
        pickle.dump(loaded_experiments, f)
    print(f"Saved {len(loaded_experiments)} experiments " f"to {summary_path}")

{'tasks': ['contextdelaydm1-ry', 'contextdelaydm1-ry'], 'paths': [PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/similarity/baseline_contextdelaydm1-ry_contextdelaydm1-ry/25100539/1/f0aaf813bc302975'), PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/similarity/baseline_contextdelaydm1-ry_contextdelaydm1-ry/25100539/42/60a59487df1add0e'), PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/similarity/baseline_contextdelaydm1-ry_contextdelaydm1-ry/25100539/128/e74498f3c9316246'), PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/similarity/baseline_contextdelaydm1-ry_contextdelaydm1-ry/25100539/128128/3cfaa882659484a4')]}
{'tasks': ['contextdelaydm2-ry', 'contextdelaydm2-ry'], 'paths': [PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/similarity/baseline_contextdelaydm2-ry_contextdelaydm2-ry/25100629/1/f0aaf813bc302975'), PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/similarity/baseline_contextdelaydm2-ry_contextdelaydm2

In [4]:
channel_to_index = {task_name: index for index, task_name in enumerate(CHANNEL)}

# row = post, column = pre
pairwise_matrix = np.empty(
    (len(CHANNEL), len(CHANNEL)),
    dtype=object,
)
pairwise_matrix[:] = None

for experiment in loaded_experiments:
    pre_task, post_task = experiment["tasks"]

    # "fdgo-ry" -> "fdgo"
    pre_task = pre_task.removesuffix("-ry")
    post_task = post_task.removesuffix("-ry")

    col = channel_to_index[pre_task]
    row = channel_to_index[post_task]

    pairwise_matrix[row, col] = experiment["data"]

In [5]:
def plot_pairwise_heatmap(
    ax,
    pairwise_matrix,
    apply_fn,
    title=None,
    show_xlabel=False,
    show_ylabel=False,
    show_colorbar=False,
    percentage=False,
    display="mean",
):
    valid_display = {"mean", "std"}
    if display not in valid_display:
        raise ValueError(f"display must be one of {valid_display}, got {display!r}.")

    mean_values = np.full(
        pairwise_matrix.shape,
        np.nan,
        dtype=float,
    )
    std_values = np.full(
        pairwise_matrix.shape,
        np.nan,
        dtype=float,
    )

    # ================= compute mean/std =================
    for row in range(pairwise_matrix.shape[0]):
        for col in range(pairwise_matrix.shape[1]):
            element = pairwise_matrix[row, col]

            if element is None:
                continue

            data = apply_fn(element)
            data = np.asarray(data, dtype=float).ravel()
            data = data[np.isfinite(data)]

            if data.size == 0:
                continue

            mean_values[row, col] = np.mean(data)
            std_values[row, col] = np.std(data)

    # ================= select displayed values =================
    if display == "mean":
        displayed_values = mean_values.copy()
    else:
        displayed_values = std_values.copy()

    if percentage:
        displayed_values *= 100

    # ================= build annotations =================
    annotations = np.full(
        pairwise_matrix.shape,
        "",
        dtype=object,
    )

    for row in range(pairwise_matrix.shape[0]):
        for col in range(pairwise_matrix.shape[1]):
            value = displayed_values[row, col]

            if not np.isfinite(value):
                continue

            annotations[row, col] = f"{value:.0f}"

    # ================= color range =================
    color_values = displayed_values.copy()

    # 对角线数值不参与颜色范围计算
    np.fill_diagonal(color_values, np.nan)

    valid_values = color_values[np.isfinite(color_values)]

    if valid_values.size == 0:
        raise ValueError(f"No valid off-diagonal values found for {title!r}.")

    if percentage:
        vmin = 0
        vmax = 100
    else:
        vmin = 0
        vmax = valid_values.max()

        if np.isclose(vmax, 0.0):
            vmax = 1.0

    plot_values = displayed_values[::-1, :]
    plot_annotations = annotations[::-1, :]

    # ================= colorbar axis =================
    if show_colorbar:
        divider = make_axes_locatable(ax)
        cax = divider.append_axes(
            "right",
            size="3.5%",
            pad=0.08,
        )
    else:
        cax = None

    # ================= heatmap =================
    n = plot_values.shape[0]
    diagonal_mask = np.eye(n, dtype=bool)[::-1, :]
    heatmap_mask = np.isnan(plot_values) | diagonal_mask

    # ================= non-diagonal heatmap =================
    heatmap = sns.heatmap(
        plot_values,
        ax=ax,
        cbar=show_colorbar,
        cbar_ax=cax,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        mask=heatmap_mask,
        xticklabels=[CHANNEL_NAME_MAPPING[task] for task in CHANNEL],
        yticklabels=[CHANNEL_NAME_MAPPING[task] for task in CHANNEL[::-1]],
        square=True,
        annot=np.where(diagonal_mask, "", plot_annotations),
        fmt="",
        linewidths=0.25,
        linecolor="white",
    )

    if show_colorbar:
        cax.yaxis.set_major_formatter(FormatStrFormatter("%.0f"))

    # ================= diagonal reference cells =================
    for original_index in range(n):
        plot_row = n - 1 - original_index
        plot_col = original_index

        annotation = plot_annotations[plot_row, plot_col]

        if annotation == "":
            continue

        ax.add_patch(
            Rectangle(
                (plot_col, plot_row),
                1,
                1,
                facecolor="lightgray",
                edgecolor="white",
                linewidth=0.25,
            )
        )

        ax.text(
            plot_col + 0.5,
            plot_row + 0.5,
            annotation,
            ha="center",
            va="center",
        )

    # ================= style =================
    ax.set_title(title, pad=12, fontsize=subtitle_fontsize)

    if show_xlabel:
        ax.set_xlabel("Task 1")
        ax.tick_params(
            axis="x",
            labelrotation=45,
        )
    else:
        ax.tick_params(
            axis="x",
            which="both",
            bottom=False,
            labelbottom=False,
        )

    if show_ylabel:
        ax.set_ylabel("Task 2")
        ax.tick_params(
            axis="y",
            labelrotation=0,
        )
    else:
        ax.tick_params(
            axis="y",
            which="both",
            left=False,
            labelleft=False,
        )

    plt.setp(
        ax.get_xticklabels(),
        horizontalalignment="right",
        rotation_mode="anchor",
    )

    return {
        "mean": mean_values,
        "std": std_values,
        "displayed": displayed_values,
    }

In [6]:
def plot_figures(prefix="Mean", display="mean", title="Figure4"):
    fig = plt.figure(figsize=(20, 20))

    gs = fig.add_gridspec(
        11,
        10,
        height_ratios=[
            1,
            1,
            1,
            1,
            1,  # top panel
            1.25,  # spacer row
            1,
            1,
            1,
            1,
            1,  # bottom panels
        ],
        hspace=0,
        wspace=0,
    )

    ax_main = fig.add_subplot(gs[0:5, 2:8])

    ax_left = fig.add_subplot(
        gs[6:11, 0:5],
        sharex=ax_main,
        sharey=ax_main,
    )

    ax_right = fig.add_subplot(
        gs[6:11, 5:10],
        sharex=ax_main,
        sharey=ax_main,
    )

    axes = [ax_main, ax_left, ax_right]

    # plots
    results = plot_pairwise_heatmap(
        axes[0],
        pairwise_matrix,
        apply_fn=lambda data: data["extra_neuron"],
        title=f"{prefix} Additional Neurons Recruited",
        show_xlabel=True,
        show_ylabel=True,
        show_colorbar=True,
        display=display,
    )

    plot_pairwise_heatmap(
        axes[1],
        pairwise_matrix,
        apply_fn=lambda data: data["accuracy1"],
        title=f"{prefix} Preserved Task 1 Accuracy (%)",
        show_xlabel=True,
        show_ylabel=True,
        percentage=True,
        display=display,
    )

    plot_pairwise_heatmap(
        axes[2],
        pairwise_matrix,
        apply_fn=lambda data: data["accuracy2"],
        title=f"{prefix} Learned Task 2 Accuracy (%)",
        show_xlabel=True,
        show_colorbar=True,
        percentage=True,
        display=display,
    )

    for label, ax in zip("abc", axes):
        ax.text(
            -0.02,
            1.02,
            f"{label}",
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=panel_indexing_fontsize,
            fontweight="bold",
            clip_on=False,
        )

    fig.savefig(
        out_path / f"{title}.png",
        dpi=300,
        bbox_inches="tight",
    )
    # fig.savefig(
    #     out_path / "Figure4.svg",
    #     bbox_inches="tight",
    # )

    plt.close(fig)
    return results

In [7]:
results = plot_figures(prefix="Mean", display="mean")
plot_figures(prefix="SD of", display="std", title="SupplementaryFigure3")
extra_mean = results["mean"]

In [8]:
import numpy as np


def optimize_task_order(
    extra_mean,
    task_names,
    objective="min",
):
    """
    在 pairwise 一阶近似下，寻找相邻任务转换的累计 extra 最优顺序。

    Parameters
    ----------
    extra_mean : array-like, shape (n_tasks, n_tasks)
        extra_mean[post, pre] 表示：
        已学习 pre 后，再学习 post 所需的预计新增 neuron 数量。

    task_names : sequence of str
        任务名称，顺序必须与 extra_mean 的两个轴一致。

    objective : {"min", "max"}
        "min"：寻找累计 extra 最小的顺序。
        "max"：寻找累计 extra 最大的顺序。

    Returns
    -------
    result : dict
        {
            "objective": str,
            "order": list[str],
            "order_indices": list[int],
            "transition_costs": list[float],
            "total_cost": float,
        }
    """
    extra_mean = np.asarray(extra_mean, dtype=np.float64)
    task_names = list(task_names)

    n_tasks = len(task_names)

    if extra_mean.shape != (n_tasks, n_tasks):
        raise ValueError(
            "extra_mean 的形状必须是 "
            f"({n_tasks}, {n_tasks})，当前为 {extra_mean.shape}。"
        )

    if objective not in {"min", "max"}:
        raise ValueError(f"objective 必须是 'min' 或 'max'，当前为 {objective!r}。")

    # transition_cost[pre, post]
    #
    # 原始矩阵定义：
    # extra_mean[post, pre]
    #
    # 转置后：
    # transition_cost[pre, post]
    transition_cost = extra_mean.T.copy()

    # 同一任务不能转移到自身。
    # 实际上 DP 中不会重复访问任务，但这里保留明确语义。
    if objective == "min":
        invalid_value = np.inf
        initial_value = np.inf

        def is_better(candidate, current):
            return candidate < current

        def select_final(costs):
            return int(np.argmin(costs))

    else:
        invalid_value = -np.inf
        initial_value = -np.inf

        def is_better(candidate, current):
            return candidate > current

        def select_final(costs):
            return int(np.argmax(costs))

    np.fill_diagonal(transition_cost, invalid_value)

    n_states = 1 << n_tasks

    # dp[mask, last]:
    # 已访问 mask 中所有任务，并以 last 结尾时的最优累计代价。
    dp = np.full(
        (n_states, n_tasks),
        initial_value,
        dtype=np.float64,
    )

    # parent[mask, last]:
    # 对应最优路径中 last 前面的任务。
    parent = np.full(
        (n_states, n_tasks),
        -1,
        dtype=np.int16,
    )

    # 任意任务都可以作为起点。
    # 起点不产生 pairwise extra cost。
    for start in range(n_tasks):
        dp[1 << start, start] = extra_mean[start][start]

    # Held–Karp 动态规划
    for mask in range(n_states):
        for last in range(n_tasks):
            current_cost = dp[mask, last]

            # 对于 min，未访问状态是 +inf；
            # 对于 max，未访问状态是 -inf。
            if not np.isfinite(current_cost):
                continue

            for next_task in range(n_tasks):
                # 每个任务只能访问一次。
                if mask & (1 << next_task):
                    continue

                edge_cost = transition_cost[last, next_task]

                # 跳过 NaN 或无穷代价的无效边。
                if not np.isfinite(edge_cost):
                    continue

                next_mask = mask | (1 << next_task)
                candidate_cost = current_cost + edge_cost

                if is_better(
                    candidate_cost,
                    dp[next_mask, next_task],
                ):
                    dp[next_mask, next_task] = candidate_cost
                    parent[next_mask, next_task] = last

    full_mask = n_states - 1

    # 完整路径可以以任意任务结尾。
    final_task = select_final(dp[full_mask])
    total_cost = float(dp[full_mask, final_task])

    if not np.isfinite(total_cost):
        raise RuntimeError(
            f"No complete valid task order was found for objective={objective!r}."
        )

    # 回溯最优路径。
    order_indices_reversed = []

    mask = full_mask
    current = final_task

    while current != -1:
        order_indices_reversed.append(current)

        previous = int(parent[mask, current])

        mask ^= 1 << current
        current = previous

    order_indices = order_indices_reversed[::-1]

    if len(order_indices) != n_tasks:
        raise RuntimeError("Recovered path does not contain every task exactly once.")

    order = [task_names[index] for index in order_indices]

    transition_costs = [
        float(
            extra_mean[
                order_indices[i + 1],  # post
                order_indices[i],  # pre
            ]
        )
        for i in range(n_tasks - 1)
    ]

    return {
        "objective": objective,
        "order": order,
        "order_indices": order_indices,
        "transition_costs": transition_costs,
        "total_cost": total_cost,
    }

In [9]:
minimum_extra_order = optimize_task_order(
    extra_mean=extra_mean,
    task_names=task_names,
    objective="min",
)

maximum_extra_order = optimize_task_order(
    extra_mean=extra_mean,
    task_names=task_names,
    objective="max",
)

In [10]:
print(minimum_extra_order)
print(maximum_extra_order)

{'objective': 'min', 'order': ['reactanti', 'reactgo', 'dm2', 'contextdm2', 'multidm', 'contextdm1', 'dm1', 'contextdelaydm1', 'delaydm1', 'multidelaydm', 'contextdelaydm2', 'delaydm2', 'delaygo', 'delayanti', 'fdanti', 'fdgo'], 'order_indices': [4, 1, 7, 9, 10, 8, 6, 13, 11, 15, 14, 12, 2, 5, 3, 0], 'transition_costs': [6.5, 44.75, 5.25, 19.0, 4.5, 3.75, 11.25, 6.0, 5.25, 10.5, 10.25, 12.75, 7.25, 3.25, 7.75], 'total_cost': 174.25}
{'objective': 'max', 'order': ['dm2', 'delayanti', 'delaydm2', 'dm1', 'reactanti', 'contextdelaydm1', 'multidm', 'reactgo', 'multidelaydm', 'delaygo', 'delaydm1', 'fdanti', 'contextdm1', 'contextdelaydm2', 'fdgo', 'contextdm2'], 'order_indices': [7, 5, 12, 6, 4, 13, 10, 1, 15, 2, 11, 3, 8, 14, 0, 9], 'transition_costs': [16.5, 48.5, 45.75, 14.25, 45.25, 41.75, 14.5, 42.0, 16.5, 54.75, 17.75, 80.5, 19.25, 17.5, 99.25], 'total_cost': 695.0}
